# 03 · Cómo ejecuta Spark el análisis

**Pregunta central:** ¿cómo transforma Spark las operaciones que escribimos en una ejecución distribuida?

Se siguen consultas reales de `02_analysis.ipynb` desde el logical plan, pasando por el optimizador Catalyst
y el physical plan, hasta los jobs, stages y tasks que se ejecutan en los cores.

| Sección | Consulta | Requisito del proyecto |
|---|---|---|
| 1 | Q2, distritos con mayor incidencia, en DataFrame API **y** en Spark SQL | 8 (ambas APIs, mismo resultado) · 9 (plan 1) |
| 2 | Q5, casos en distritos en emergencia por lluvias (`left_semi` join) | 9 (plan 2) |
| 3 | Q1, casos por año: de las transformations a los jobs, stages y tasks | 11 (lazy evaluation, DAG, stages, tasks) |
| 4 | Los shuffles de estas consultas, medidos | 10 (shuffle) |

Spark corre en local mode: una sola JVM donde el *driver* planifica el trabajo y los threads *executor* (uno por
core) ejecutan las tasks. Los mismos planes correrían sin cambios en un cluster.

In [1]:
import contextlib
import io
import json
import re
import time
import urllib.request
from pathlib import Path

from pyspark.sql import SparkSession, Window, functions as F

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "catalog.yaml").exists())
PROCESSED = ROOT / "data" / "processed"

spark = (SparkSession.builder
         .master("local[*]")
         .appName("dengue-execution-plans")
         .config("spark.sql.session.timeZone", "America/Lima")
         .config("spark.sql.maxMetadataStringLength", 1000)  # don't truncate file paths and filters in plans
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

cases, population, emergencies = (spark.read.parquet(str(PROCESSED / name))
                                  for name in ["dengue_cases", "population", "water_emergencies"])

for key in ["spark.sql.shuffle.partitions", "spark.sql.adaptive.enabled", "spark.sql.autoBroadcastJoinThreshold"]:
    print(f"{key:40} {spark.conf.get(key)}")
print(f"{'cores (default parallelism)':40} {sc.defaultParallelism}")
print(f"{'Spark UI':40} {sc.uiWebUrl}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/01 20:07:38 WARN Utils: Your hostname, Raymonds-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.18.58 instead (on interface en0)
26/11/01 20:07:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/01 20:07:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


spark.sql.shuffle.partitions             200
spark.sql.adaptive.enabled               true
spark.sql.autoBroadcastJoinThreshold     10485760b
cores (default parallelism)              12
Spark UI                                 http://192.168.18.58:4040


In [2]:
def plan_text(df, mode="formatted"):
    # The text that df.explain(mode) prints, captured as a string.
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode=mode)
    return buffer.getvalue()


def plan_section(df, name):
    # One section of explain("extended"): "Parsed Logical Plan", "Optimized Logical Plan", "Physical Plan"...
    return plan_text(df, "extended").split(f"== {name} ==\n")[1].split("\n== ")[0]


def without_ids(text):
    # Expression ids (cases#23L) and plan ids differ between two separately built plans.
    return re.sub(r"#\d+L?|plan_id=\d+", "", text)


def stage_metrics(job_group):
    # Stages run for a job group, read from the Spark UI REST API (the same numbers the UI shows).
    time.sleep(1)  # let the UI listener record the last events
    base = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}"
    jobs = [j for j in json.load(urllib.request.urlopen(f"{base}/jobs")) if j.get("jobGroup") == job_group]
    print(f"{'job':>3} {'stage':>5} {'status':9} {'tasks':>5} {'input rows':>11} {'shuffle write rows':>18} "
          f"{'shuffle read rows':>17} {'shuffle write bytes':>19}")
    for job in sorted(jobs, key=lambda j: j["jobId"]):
        for stage_id in sorted(job["stageIds"]):
            s = json.load(urllib.request.urlopen(f"{base}/stages/{stage_id}"))[0]
            print(f"{job['jobId']:>3} {stage_id:>5} {s['status']:9} {s['numTasks']:>5} {s['inputRecords']:>11,} "
                  f"{s['shuffleWriteRecords']:>18,} {s['shuffleReadRecords']:>17,} {s['shuffleWriteBytes']:>19,}")

## 1 · Plan 1: la misma consulta en DataFrame API y en Spark SQL

Q2: para cada año 2022–2025, los cinco distritos (10,000+ habitantes) con mayor incidencia. Ambas versiones
expresan la misma lógica: contar casos por distrito y año, hacer join con la población, calcular un ranking con
una window function y ordenar.

In [3]:
MIN_POPULATION = 10_000

district_cases = (cases.filter(F.col("year").between(2022, 2025))
                  .groupBy("year", "ubigeo").agg(F.count("*").alias("cases")))
district_population = (population.groupBy("year", "ubigeo", "department", "province", "district")
                       .agg(F.sum("population").alias("population")))
by_year = Window.partitionBy("year").orderBy(F.desc(F.col("cases") / F.col("population")))

top_districts_df = (district_cases
                    .join(district_population, ["year", "ubigeo"], "left")
                    .filter(F.col("population") >= MIN_POPULATION)
                    .select("year", "department", "province", "district", "cases", "population",
                            F.round(F.col("cases") / F.col("population") * 100000, 1).alias("incidence_100k"),
                            F.dense_rank().over(by_year).alias("rank"))
                    .filter(F.col("rank") <= 5)
                    .select("year", "rank", "department", "province", "district", "cases", "population", "incidence_100k")
                    .orderBy("year", "rank"))

In [4]:
# Spark SQL needs the tables registered as views
cases.createOrReplaceTempView("dengue_cases")
population.createOrReplaceTempView("population")

top_districts_sql = spark.sql(f"""
    WITH district_cases AS (
        SELECT year, ubigeo, COUNT(*) AS cases
        FROM dengue_cases
        WHERE year BETWEEN 2022 AND 2025
        GROUP BY year, ubigeo
    ),
    district_population AS (
        SELECT year, ubigeo, department, province, district, SUM(population) AS population
        FROM population
        GROUP BY year, ubigeo, department, province, district
    ),
    ranked AS (
        SELECT year, department, province, district, cases, population,
               ROUND(cases / population * 100000, 1) AS incidence_100k,
               DENSE_RANK() OVER (PARTITION BY year ORDER BY cases / population DESC) AS rank
        FROM district_cases
        LEFT JOIN district_population USING (year, ubigeo)
        WHERE population >= {MIN_POPULATION}
    )
    SELECT year, rank, department, province, district, cases, population, incidence_100k
    FROM ranked
    WHERE rank <= 5
    ORDER BY year, rank
""")

### Mismo resultado

`exceptAll` en ambas direcciones devuelve las filas que faltan en el otro resultado (incluidos duplicados).
Ambos deben estar vacíos.

In [5]:
only_in_df = top_districts_df.exceptAll(top_districts_sql).count()
only_in_sql = top_districts_sql.exceptAll(top_districts_df).count()
print(f"rows: DataFrame API {top_districts_df.count()}, Spark SQL {top_districts_sql.count()}")
print(f"only in DataFrame API: {only_in_df} | only in Spark SQL: {only_in_sql}")
assert only_in_df == 0 and only_in_sql == 0
top_districts_sql.show(20, truncate=False)

rows: DataFrame API 20, Spark SQL 20
only in DataFrame API: 0 | only in Spark SQL: 0


+----+----+-----------+-----------------+--------------------+-----+----------+--------------+
|year|rank|department |province         |district            |cases|population|incidence_100k|
+----+----+-----------+-----------------+--------------------+-----+----------+--------------+
|2022|1   |ÁNCASH     |CASMA            |CASMA               |1927 |38179     |5047.3        |
|2022|2   |PIURA      |MORROPÓN         |MORROPÓN            |683  |14420     |4736.5        |
|2022|3   |AMAZONAS   |BAGUA            |BAGUA               |1342 |29254     |4587.4        |
|2022|4   |CUSCO      |LA CONVENCIÓN    |SANTA ANA           |1775 |41669     |4259.8        |
|2022|5   |JUNÍN      |CHANCHAMAYO      |SAN RAMÓN           |1110 |29710     |3736.1        |
|2023|1   |PIURA      |PAITA            |LA HUACA            |1890 |13958     |13540.6       |
|2023|2   |ÁNCASH     |SANTA            |NEPEÑA              |1428 |15854     |9007.2        |
|2023|3   |PIURA      |SULLANA          |MIGUEL CH

### Mismo plan

Los planes se comparan después de quitar los expression ids internos (`cases#23L`), que cambian cada vez que se
construye un plan.

In [6]:
for name in ["Parsed Logical Plan", "Analyzed Logical Plan", "Optimized Logical Plan", "Physical Plan"]:
    same = without_ids(plan_section(top_districts_df, name)) == without_ids(plan_section(top_districts_sql, name))
    print(f"{name:24} identical: {same}")
same_formatted = without_ids(plan_text(top_districts_df)) == without_ids(plan_text(top_districts_sql))
label = 'explain("formatted")'
print(f"{label:24} identical: {same_formatted}")
assert same_formatted

Parsed Logical Plan      identical: False
Analyzed Logical Plan    identical: False
Optimized Logical Plan   identical: True
Physical Plan            identical: True
explain("formatted")     identical: True


**Mismo resultado y mismo plan.** El parsed logical plan y el analyzed logical plan difieren: el texto SQL se convierte en un árbol con CTEs y un join `USING`, mientras que la DataFrame API arma el árbol a partir de llamadas a métodos. Desde el **optimized logical plan** en adelante, incluido el physical plan, son idénticos: a Catalyst no le importa con qué API se escribió la consulta.

### Logical plan después de Catalyst

El optimized logical plan (de `explain("extended")`), antes de que Spark decida *cómo* ejecutarlo:

In [7]:
print(plan_section(top_districts_sql, "Optimized Logical Plan"))

Sort [year#1351 ASC NULLS FIRST, rank#1367 ASC NULLS FIRST], true
+- Project [year#1351, rank#1367, department#1355, province#1356, district#1357, cases#1352L, population#1361L, incidence_100k#1362]
   +- Filter (rank#1367 <= 5)
      +- Window [dense_rank(_w0#1363) windowspecdefinition(year#1351, _w0#1363 DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS rank#1367], [year#1351], [_w0#1363 DESC NULLS LAST]
         +- WindowGroupLimit [year#1351], [_w0#1363 DESC NULLS LAST], dense_rank(_w0#1363), 5
            +- Project [year#1351, department#1355, province#1356, district#1357, cases#1352L, population#1361L, round(((cast(cases#1352L as double) / cast(population#1361L as double)) * 100000.0), 1) AS incidence_100k#1362, (cast(cases#1352L as double) / cast(population#1361L as double)) AS _w0#1363]
               +- Join Inner, ((year#1351 = year#1353) AND (ubigeo#1339 = ubigeo#1354))
                  :- Aggregate [year#1351, ubigeo#1339], [year#13

### Physical plan

In [8]:
top_districts_df.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (25)
+- Sort (24)
   +- Exchange (23)
      +- Project (22)
         +- Filter (21)
            +- Window (20)
               +- WindowGroupLimit (19)
                  +- Sort (18)
                     +- Exchange (17)
                        +- WindowGroupLimit (16)
                           +- Sort (15)
                              +- Project (14)
                                 +- BroadcastHashJoin Inner BuildRight (13)
                                    :- HashAggregate (5)
                                    :  +- Exchange (4)
                                    :     +- HashAggregate (3)
                                    :        +- Filter (2)
                                    :           +- Scan parquet  (1)
                                    +- BroadcastExchange (12)
                                       +- Filter (11)
                                          +- HashAggregate (10)
                                             +- 

### Lectura del plan 1

**Logical plan.** Contar casos y sumar población por (año, distrito), unirlos con LEFT JOIN, quedarse con los distritos de 10,000+ habitantes, calcular el ranking dentro de cada año, quedarse con el top 5 y ordenar.

**Qué cambió Catalyst:**

1. **Outer → inner join** (`Join Inner`). `population >= 10000` elimina las filas con nulos que agrega el LEFT JOIN, así que ambos dan el mismo resultado.
2. **Inferred filters.** El rango de años escrito para los casos llega también a la población a través del join key: `PushedFilters: [GreaterThanOrEqual(year,2022), LessThanOrEqual(year,2025), ...]`.
3. **Predicate pushdown.** `population >= 10000` se ejecuta antes del join (operador 11).
4. **Partition pruning.** `PartitionFilters: [... year >= 2022, year <= 2025 ...]`: solo se abren 4 de las 26 carpetas de años.
5. **Dynamic partition pruning.** `dynamicpruningexpression(...)` agrega en tiempo de ejecución los años presentes en el lado de la población como un filtro más sobre las carpetas de casos.
6. **Column pruning.** El scan de casos lee `struct<ubigeo:string>`, 1 de 14 columnas (el año sale del nombre de la carpeta).
7. **Top-N antes del shuffle.** `WindowGroupLimit` conserva los 5 mejores distritos por año antes del shuffle del `Window` (Partial, operador 16) y después (Final, operador 19).

**Physical plan:**

- **Agregación en dos fases:** `HashAggregate (partial_count)` → `Exchange hashpartitioning(year, ubigeo, 200)` → `HashAggregate (count)`, para casos y para población (operadores 6–10).
- **Broadcast join:** la población filtrada (unos miles de filas) está por debajo de los 10 MB de `autoBroadcastJoinThreshold`, así que se copia a cada task (`BroadcastHashJoin Inner BuildRight`) y los casos no pasan por un shuffle para el join.
- **Window:** `Exchange hashpartitioning(year)` → `Sort` → `Window` (`dense_rank`) → `Filter (rank <= 5)`.
- **Global sort:** `Exchange rangepartitioning(year, rank)` → `Sort`.
- **AQE:** `AdaptiveSparkPlan isFinalPlan=false`: el plan aún puede cambiar cuando se conozcan los tamaños reales.

En total: **cuatro shuffles** (los operadores `Exchange`) y **un broadcast**.

## 2 · Plan 2: un semi join que no necesita shuffle

Q5 contó los casos de 2023–2024 en distritos en emergencia por lluvias. Un distrito aparece una vez por cada
decreto y prórroga, así que un inner join repite cada caso; un `left_semi` join solo verifica que exista una
coincidencia.

In [9]:
YEARS = [2023, 2024]
cases_2324 = cases.filter(F.col("year").isin(YEARS)).select("year", "ubigeo")
rain_decrees = (emergencies.filter((F.col("emergency_type") == "rain") & F.col("year").isin(YEARS))
                .select("year", "ubigeo"))

semi = cases_2324.join(rain_decrees, ["year", "ubigeo"], "left_semi")
inner = cases_2324.join(rain_decrees, ["year", "ubigeo"], "inner")
print(f"left_semi: {semi.count():>9,} rows | inner: {inner.count():>9,} rows | cases 2023-2024: {cases_2324.count():>9,}")
semi.explain("formatted")

left_semi:   352,213 rows | inner: 1,539,044 rows | cases 2023-2024:   528,172
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- BroadcastHashJoin LeftSemi BuildRight (7)
      :- Filter (2)
      :  +- Scan parquet  (1)
      +- BroadcastExchange (6)
         +- Project (5)
            +- Filter (4)
               +- Scan parquet  (3)


(1) Scan parquet 
Output [2]: [ubigeo#1, year#13]
Batched: true
Location: InMemoryFileIndex [file:/Users/raymondnegron/Documents/MAE - Data Science/MAE-CDE-apache-spark/data/processed/dengue_cases]
PartitionFilters: [year#13 IN (2023,2024), isnotnull(year#13), dynamicpruningexpression(year#13 IN dynamicpruning#1394)]
PushedFilters: [IsNotNull(ubigeo)]
ReadSchema: struct<ubigeo:string>

(2) Filter
Input [2]: [ubigeo#1, year#13]
Condition : isnotnull(ubigeo#1)

(3) Scan parquet 
Output [3]: [ubigeo#22, year#26, emergency_type#35]
Batched: true
Location: InMemoryFileIndex [file:/Users/raymondnegron/Documents/MAE - Data Science/MAE-CDE-apache-

In [10]:
# The inner join: same strategy, different join type
inner.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [year#13, ubigeo#1]
   +- BroadcastHashJoin [year#13, ubigeo#1], [year#26, ubigeo#22], Inner, BuildRight, false, false
      :- Filter isnotnull(ubigeo#1)
      :  +- FileScan parquet [ubigeo#1,year#13] Batched: true, DataFilters: [isnotnull(ubigeo#1)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/raymondnegron/Documents/MAE - Data Science/MAE-CDE-apache-spark/data/processed/dengue_cases], PartitionFilters: [year#13 IN (2023,2024), isnotnull(year#13), dynamicpruningexpression(year#13 IN dynamicpruning#1395)], PushedFilters: [IsNotNull(ubigeo)], ReadSchema: struct<ubigeo:string>
      :        +- SubqueryAdaptiveBroadcast dynamicpruning#1395, [0], true, Project [year#26, ubigeo#22], [year#26, ubigeo#22]
      :           +- AdaptiveSparkPlan isFinalPlan=false
      :              +- Project [year#26, ubigeo#22]
      :                 +- Filter ((((isnotnull(emergency_type#35) AND (emergency_type#35 

### Lectura del plan 2

Solo cambia el tipo de join, pero el inner join devuelve **1,539,044 filas** (casi el triple de los 528,172 casos) y el semi join, las **352,213** correctas.

**Logical plan.** Quedarse con los casos de 2023–2024 y los decretos por lluvias de 2023–2024; join por (año, distrito).

**Qué cambió Catalyst:**

- **Partition pruning:** `PartitionFilters: [year IN (2023,2024)]` lee 2 de las 26 carpetas de casos.
- **Filter pushdown:** `PushedFilters: [EqualTo(emergency_type,rain), In(year, [2023,2024]), ...]` permite que el Parquet reader salte los row groups que no pueden coincidir.
- **Column pruning:** los casos leen solo `ubigeo`; los decretos, 3 de sus 15 columnas.
- **Dynamic partition pruning:** los años que quedan en el lado de los decretos filtran las carpetas de casos en tiempo de ejecución.

**Physical plan: sin shuffle.** Los decretos filtrados son pocos, así que Spark hace broadcast de ellos a todas las tasks como una hash table (`BroadcastHashJoin LeftSemi BuildRight`). Cada task revisa sus propios casos contra esa hash table y emite cada caso como máximo una vez, **sin `Exchange`**. El inner join usa la misma estrategia, pero emite una fila por cada decreto que coincide, y por eso su conteo se infla.

## 3 · De las transformations a los jobs, stages y tasks

La agregación de Q1 (casos por año), marcada con un job group para rastrear lo que Spark ejecuta para ella.

In [11]:
sc.setJobGroup("q1", "cases per year")

start = time.perf_counter()
cases_per_year = (cases.groupBy("year")
                  .agg(F.count("*").alias("cases"))
                  .orderBy("year"))
print(f"transformations defined in {time.perf_counter() - start:.3f} s")
print("jobs run so far:", sc.statusTracker().getJobIdsForGroup("q1"))

transformations defined in 0.008 s
jobs run so far: []


**Lazy evaluation.** Definir la consulta tomó milisegundos y no ejecutó **ningún job**: `groupBy`, `agg` y `orderBy` son *transformations* que solo amplían el logical plan. No se lee nada hasta que una *action* (`collect`, `count`, `show`, `write`) pide un resultado.

In [12]:
start = time.perf_counter()
rows = cases_per_year.collect()  # the action
print(f"collect() returned {len(rows)} rows in {time.perf_counter() - start:.2f} s")
print("jobs run:", sorted(sc.statusTracker().getJobIdsForGroup("q1")))
stage_metrics("q1")

collect() returned 26 rows in 0.14 s
jobs run: [48, 49, 50, 51]


job stage status    tasks  input rows shuffle write rows shuffle read rows shuffle write bytes
 48   106 COMPLETE      9   1,065,274                 26                 0               1,664
 49   107 SKIPPED       9           0                  0                 0                   0
 49   108 COMPLETE      1           0                  0                26                   0
 50   109 SKIPPED       9           0                  0                 0                   0
 50   110 COMPLETE      1           0                 26                26               1,664
 51   111 SKIPPED       9           0                  0                 0                   0
 51   112 SKIPPED       1           0                  0                 0                   0
 51   113 COMPLETE      1           0                  0                26                   0


In [13]:
# After execution the plan shows what adaptive query execution (AQE) decided at runtime
cases_per_year.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (18)
+- == Final Plan ==
   ResultQueryStage (12)
   +- * Sort (11)
      +- AQEShuffleRead (10), coalesced
         +- ShuffleQueryStage (9), Statistics(sizeInBytes=624.0 B, rowCount=26)
            +- Exchange (8)
               +- * HashAggregate (7)
                  +- AQEShuffleRead (6), coalesced
                     +- ShuffleQueryStage (5), Statistics(sizeInBytes=624.0 B, rowCount=26)
                        +- Exchange (4)
                           +- * HashAggregate (3)
                              +- * ColumnarToRow (2)
                                 +- Scan parquet  (1)
+- == Initial Plan ==
   Sort (17)
   +- Exchange (16)
      +- HashAggregate (15)
         +- Exchange (14)
            +- HashAggregate (13)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [year#13]
Batched: true
Location: InMemoryFileIndex [file:/Users/raymondnegron/Documents/MAE - Data Science/MAE-CDE-apache-spark/data/processed/dengue_cases]

**Del plan al DAG, stages y tasks.** La action convierte el physical plan en un **DAG de stages**, cortado en cada `Exchange`: un stage solo puede empezar cuando sus datos de entrada ya pasaron por el shuffle. Esta consulta tiene dos exchanges (`groupBy` y `orderBy`):

| Stage (en la tabla de arriba) | Tasks | Qué hace |
|---|---|---|
| primero | 9 | Lee los 1,065,274 casos (sin columnas de datos: `ReadSchema: struct<>`, el año es el nombre de la carpeta), cuenta por año en cada task (`partial_count`) y escribe **26 filas parciales** al shuffle |
| siguiente | 1 | **Muestrea las sort keys**: `orderBy` necesita los límites de cada rango antes de redistribuir |
| siguiente | 1 | Termina el conteo por año y escribe los 26 totales con `rangepartitioning` para el sort |
| último | 1 | Hace el sort de cada rango y devuelve las 26 filas al driver |

- **Tasks:** una por partition: 9 para el scan (los archivos Parquet agrupados en 9 splits para 12 cores) y 1 después.
- **Varios jobs por action:** con AQE cada stage con shuffle es su propio job, así que un `collect()` generó 4 jobs; los stages ya calculados aparecen como **SKIPPED** y reutilizan sus shuffle files.
- **Final plan:** muestra lo que AQE decidió en tiempo de ejecución. `ShuffleQueryStage` reporta tamaños reales (624 bytes, 26 filas) y `AQEShuffleRead coalesced` junta las 200 shuffle partitions en 1. Los operadores marcados con `*` comparten un `codegen id`: **whole-stage code generation** compila cada cadena de operadores en una sola función Java.

El DAG se puede ver en la Spark UI que imprime la celda de configuración, si el notebook se ejecuta de forma interactiva (**Jobs → DAG Visualization**; la pestaña **SQL / DataFrame** muestra el plan con el número de filas por operador).

## 4 · Shuffle: por qué hay que mover los datos y cuánto se mueve

La misma agregación de Q1, ahora con adaptive query execution (AQE) desactivado, para ver el shuffle con las
200 partitions por defecto de Spark.

In [14]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
sc.setJobGroup("q1-no-aqe", "cases per year without AQE")

start = time.perf_counter()
cases.groupBy("year").agg(F.count("*").alias("cases")).orderBy("year").collect()
print(f"without AQE: {time.perf_counter() - start:.2f} s")
stage_metrics("q1-no-aqe")

spark.conf.set("spark.sql.adaptive.enabled", "true")
sc.setJobGroup("other", "rest of the notebook")

without AQE: 0.24 s


job stage status    tasks  input rows shuffle write rows shuffle read rows shuffle write bytes
 52   114 COMPLETE      9   1,065,274                 26                 0               1,664
 52   115 COMPLETE    200           0                  0                26                   0
 53   116 SKIPPED       9           0                  0                 0                   0
 53   117 COMPLETE    200           0                 26                26               1,664
 53   118 COMPLETE     26           0                  0                26                   0


**Por qué hace falta el shuffle.** Los casos de cada año están repartidos en archivos que leen tasks distintas. Para obtener un total por año, todos los conteos parciales de un año deben juntarse en la **misma partition**: `Exchange hashpartitioning(year)` envía cada fila a la partition `hash(year) mod N`. Cada task escribe su salida en disco local, agrupada por partition de destino (*shuffle write*), y el stage siguiente recoge su parte de cada una (*shuffle read*). `orderBy` hace lo mismo con `rangepartitioning`, para que cada partition tenga un rango de keys. El shuffle es el paso más costoso (disco, serialización y, en un cluster, red) y marca el límite entre stages.

**Qué muestran las mediciones:**

- **La agregación parcial reduce el shuffle:** cada task cuenta por año antes del shuffle, así que 1,065,274 filas de entrada se convierten en **26 filas en el shuffle (1,664 bytes)**.
- **200 partitions son demasiadas aquí:** sin AQE, **200 tasks** procesan 26 filas, dos veces (para muestrear los rangos del sort y para terminar el conteo), más 26 para el sort: casi todas vacías. Con AQE, Spark midió el shuffle de 624 bytes y lo **redujo con coalesce a 1 task**.
- **El plan 1 tiene cuatro shuffles** (dos agregaciones, el window y el sort), pero ninguno para su join, gracias al broadcast. **El plan 2 no tiene ninguno.** Evitar shuffles es la principal palanca para acelerar un job de Spark.

## Resumen: cómo transforma Spark nuestras operaciones en una ejecución distribuida

Seguir estas consultas de principio a fin responde la pregunta central:

1. **Describimos el resultado, no los pasos.** Las llamadas de la DataFrame API o el texto SQL solo construyen un *logical plan*; todavía no se ejecuta nada (lazy evaluation).
2. **Analysis.** Spark resuelve nombres y tipos contra el catalog; una columna mal escrita falla aquí, antes de leer datos. Las dos APIs solo difieren hasta este punto.
3. **Optimization (Catalyst).** Las reglas de Catalyst producen el *optimized logical plan*: filter pushdown al Parquet reader, partition y column pruning, inferred filters a través de joins, outer joins convertidos en inner joins, top-N antes del shuffle del window.
4. **Physical planning.** Spark elige *cómo* ejecutar cada operador: `HashAggregate` en dos fases, `BroadcastHashJoin` para tablas pequeñas, `Exchange` donde hay que reagrupar datos, whole-stage code generation.
5. **Scheduling.** Una action convierte el plan en un DAG de **stages**, cortado en cada `Exchange`; AQE vuelve a planificar entre stages con tamaños reales (200 shuffle partitions → 1).
6. **Execution.** Cada stage corre como **tasks** en paralelo, una por partition (9 tasks en 12 cores para el scan); el **shuffle** mueve los datos entre stages.

En resumen, Spark convierte una consulta declarativa en un plan optimizado, lo divide en stages donde los datos tienen que moverse y ejecuta cada stage como tasks en paralelo. El mismo código correría sin cambios en un cluster.

In [15]:
spark.stop()